In [ ]:
import inspect
import types
import functools

# Replace/implement abstract methods for a class named `print` if present,
# otherwise provide a useful concrete `print` class that wraps the builtin.

_builtin_print = (__builtins__['print'] if isinstance(__builtins__, dict) else __builtins__.print)

def _make_impl(name):
    def impl(self, *args, **kwargs):
        _builtin_print(f"<{self.__class__.__name__}.{name}> called with", args, kwargs)
        return None
    impl.__name__ = name
    return impl

if 'print' in globals() and isinstance(globals()['print'], type):
    Base = globals()['print']
    abstract_names = getattr(Base, '__abstractmethods__', None) or set()
    if abstract_names:
        attrs = {}
        for name in abstract_names:
            # preserve signature if possible
            attrs[name] = _make_impl(name)
        # create a concrete subclass that implements the abstract methods
        Concrete = type('print', (Base,), attrs)
        globals()['print'] = Concrete
else:
    class print:
        """A simple printable wrapper class that behaves like builtin print and
        provides common file-like methods."""
        def __init__(self, *args, **kwargs):
            self._prefix = kwargs.pop('prefix', '')
            self._suffix = kwargs.pop('suffix', '')
            self._end = kwargs.pop('end', '\n')

        def __call__(self, *args, **kwargs):
            sep = kwargs.pop('sep', ' ')
            end = kwargs.pop('end', self._end)
            _builtin_print(self._prefix + sep.join(map(str, args)) + self._suffix, end=end, **kwargs)

        def write(self, s):
            # write without automatic newline
            _builtin_print(self._prefix + str(s), end='')

        def flush(self):
            # nothing buffered here, but keep API compatible
            pass

        def close(self):
            # no resources to free, present for API compatibility
            pass

        def info(self):
            _builtin_print(f"<print wrapper prefix={self._prefix!r} suffix={self._suffix!r}>")